In [1]:
#import + session DB + subgraph storage obj


from HOGDB.graph.subgraph import Subgraph  
from HOGDB.graph.subgraph import SubgraphEdge 
from HOGDB.graph.graph_with_subgraph_storage import GraphwithSubgraphStorage
from HOGDB.graph.graph_with_tuple_storage import *
from HOGDB.graph.edge import Edge
from HOGDB.db.neo4j import Neo4jDatabase

# INIT DB

db = Neo4jDatabase()
graph = GraphwithSubgraphStorage(db)

print("YOU ARE SUCCESSFULLY CONNECTED TO THE ACTIVE DB ! ")

YOU ARE SUCCESSFULLY CONNECTED TO THE ACTIVE DB ! 


In [ ]:
def R1(graph, subgraphId: str): 


    pattern = Subgraph(
    labels=[Label("taggedComment")],  
    properties=[Property("id", str, subgraphId)]
    )

    sg = graph.get_subgraph(pattern)

    if sg is None:
        print("Subgraph introuvable")
    else:
        print("Subgraph id:", sg["id"])   # grace à __getitem__

        for node in sg.subgraph_nodes:
            print("Node Labels :", node.labels)
            print("node Properties :", node.properties)
        for edge in sg.subgraph_edges:
            print("Edge Labels :", edge.label)
            print("Edge Properties :", edge.properties)
 
 


In [ ]:
import time

nb_runs = 10
temps = []

for _ in range(nb_runs):
    debut = time.perf_counter()
    #----------------------------
    R1(graph, "tc_2061584302094_2002")
    #--------------------------------
    fin = time.perf_counter()
    
    temps.append((fin - debut) * 1000)  # conversion en ms

temps_moyen = sum(temps) / len(temps)

print(f"Temps moyen : {temps_moyen:.3f} ms")

In [ ]:
#another version of R1 using traverse_path()
def R1v2(graph, subgraphId: str):


    # Subgraph pattern 
    sub = Subgraph(
        labels=[Label("taggedComment")],
        properties=[Property("id", str, subgraphId)]
    )

    # generic node that may belong to the subgraph
    node = Node()  # no labels so it matches any node type

    # IMPORTANT !!!! il faut put node first, then subgraph
    path = Path()
    path.add(node, "n")   # node variable first
    path.add(sub, "s")    # subgraph variable second

    df = graph.traverse_path(
        paths=[path],
        # condition refers to the subgraph variable 's'
        conditions=[[f"s.id = '{subgraphId}'"]],
        return_values=["n"]
    )
    for _, row in df.iterrows():
        node = row["n"]

        print("Labels:", node.labels)

        for key in node.keys(): #.properties doesn't work
            print(key, "=", node[key])


        print("------")
    

In [ ]:
def R2(graph):
    # Query to retrieve taggedComment subgraphs
    query = """
    MATCH (sg:_subgraph:taggedComment)
    RETURN sg.id
    """

    ids = [r[0] for r in graph.db._execute_query(graph.session, query)]

    subgraphs = []
    for id in ids:
        pattern = Subgraph(
            labels=[Label("taggedComment")],
            properties=[Property("id", str, id)]
        )
        sg = graph.get_subgraph(pattern)
        if sg:
            subgraphs.append(sg)

    result = {}

    for sg in subgraphs:

        tag_ids = [
            node["id"]
            for node in sg.subgraph_nodes
            if any(str(label) == "Tag" for label in node.labels)
            and node["name"] == "Mick_Jagger"
        ]

        if tag_ids:
            result[sg["id"]] = tag_ids


    return result

In [ ]:
#another version of R2 using traverse_path()
def R2v2(graph):

    tag = Node(labels=[Label("Tag")])

    sub = Subgraph(
        labels=[Label("taggedComment")]
    )

    path = Path()
    path.add(tag, "t")   # Node first
    path.add(sub, "s")   # then Subgraph

    df = graph.traverse_path(
        paths=[path],
        conditions=[[
            "t.name = 'Mick_Jagger'"
        ]],
        return_values=["t.id"]
    )

    return df

In [ ]:
def Q5a(graph):
    query = """
    MATCH (sg:_subgraph:forumMembership)
    RETURN sg.id
    """

    ids = [r[0] for r in graph.db._execute_query(graph.session, query)]

    subgraphs = []
    for id in ids:
        pattern = Subgraph(
            labels=[Label("forumMembership")],
            properties=[Property("id", str, id)]
        )
        sg = graph.get_subgraph(pattern)
        if sg:
            subgraphs.append(sg)

    result = []

    for sg in subgraphs:
        nb_member = sg.get("nbMember")

        if nb_member is None or nb_member < 150:
            continue

        for node in sg.subgraph_nodes:
            if any(str(label) == "Person" for label in node.labels):
                result.append({
                    "id": node["id"],
                    "locationIP": node["locationIP"]
                })

    return result

In [ ]:
def Q5b(graph):
    query = """
    MATCH (sg:_subgraph:forumMembership)
    RETURN sg.id
    """

    ids = [r[0] for r in graph.db._execute_query(graph.session, query)]

    subgraphs = []
    for id in ids:
        pattern = Subgraph(
            labels=[Label("forumMembership")],
            properties=[Property("id", str, id)]
        )
        sg = graph.get_subgraph(pattern)
        if sg:
            subgraphs.append(sg)

    result = []

    for sg in subgraphs:
        nb_member = sg.get("nbMember")

        if nb_member is None or nb_member > 150:
            continue

        for node in sg.subgraph_nodes:
            if any(str(label) == "Person" for label in node.labels):
                result.append({
                    "id": node["id"],
                    "locationIP": node["locationIP"]
                })

    return result

In [ ]:
#another version using traverse_path()
def Q5av2(graph):

    person = Node(labels=[Label("Person")])

    sub = Subgraph(
        labels=[Label("forumMembership")]
    )

    path = Path()
    path.add(person, "p")   # Node first
    path.add(sub, "s")      # then Subgraph

    df = graph.traverse_path(
        paths=[path],
        conditions=[[
            "s.nbMember >150"
        ]],
        return_values=[
            "p.firstName",
            "p.lastName"
        ]
    )

    return df

In [ ]:
#another version using traverse_path()
def Q5bv2(graph):

    person = Node(labels=[Label("Person")])

    sub = Subgraph(
        labels=[Label("forumMembership")]
    )

    path = Path()
    path.add(person, "p")   # Node first
    path.add(sub, "s")      # then Subgraph

    df = graph.traverse_path(
        paths=[path],
        conditions=[[
            "s.nbMember <150"
        ]],
        return_values=[
            "p.firstName",
            "p.lastName"
        ]
    )

    return df